# 01 · Bronze: ingest raw subscription data

**Pipeline:** **bronze** → silver → gold features → churn model → ARR forecast → retention playbook

There is no production source system here, so this stage simulates one: 10,000 SaaS customers over
36 months (Jan 2022 – Dec 2024). In a real deployment this notebook would be replaced by ingestion
from billing (MRR) and product analytics (logins, tickets). Everything downstream is source-agnostic.

**How churn is simulated.** Churn is driven by a *hidden* health state rather than a hard-coded
countdown, so behavioural signals help but are imperfect:

| Mechanism | Why it matters for modelling |
|---|---|
| Customers drift into an *at-risk* state (logins fall, tickets rise, contractions) and **often recover** | False positives: not every usage drop ends in churn |
| Healthy customers have **random usage dips** (holidays, re-orgs) | More false positives |
| Healthy customers can churn with **no warning** (budget cuts, acquisitions) | Silent churn caps achievable recall |
| **Annual contracts** can only churn at renewal | Renewal timing becomes a real feature |
| Expansion, contraction, upgrades and downgrades | MRR moves; net revenue retention is not flat |

The raw feeds also get realistic defects (duplicate rows, null amounts, `-1` sentinel logins) for
the silver layer to clean.

In [1]:
%pip install -q -r ../requirements.txt

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
from revenue_pipeline import config, spark_utils

spark = spark_utils.get_spark()
print(f"Spark {spark.version} | schema: {config.SCHEMA} | Databricks: {spark_utils.on_databricks()}")

Spark 3.5.3 | schema: revenue_forecasting | Databricks: False


In [3]:
from revenue_pipeline import datagen

customers, mrr, activity = datagen.simulate()

print(f"Customers: {len(customers):,} | MRR rows: {len(mrr):,} | activity rows: {len(activity):,}")
print("\nRealised annual logo churn vs target:")
print(pd.DataFrame({"realised": datagen.realised_annual_churn(customers),
                    "target": pd.Series(config.TARGET_ANNUAL_CHURN)}).round(3).to_string())
print("\nContract mix:\n" + customers.groupby(["segment", "contract_type"]).size().unstack().to_string())

Customers: 10,000 | MRR rows: 134,875 | activity rows: 134,875

Realised annual logo churn vs target:


            realised  target
Enterprise     0.052    0.05
Mid-Market     0.113    0.12
SMB            0.226    0.22

Contract mix:
contract_type  Annual  Monthly
segment                       
Enterprise        808      176
Mid-Market       1249     1309
SMB               972     5486


In [4]:
mrr_raw, activity_raw = datagen.add_raw_defects(mrr, activity)

# Bronze keeps the feeds as delivered: string dates, duplicates and bad values included.
spark_utils.write_table(spark, customers, "bronze_customers")
spark_utils.write_table(spark, mrr_raw, "bronze_mrr")
spark_utils.write_table(spark, activity_raw, "bronze_activity")

for t in ["bronze_customers", "bronze_mrr", "bronze_activity"]:
    print(f"{t:<18} {spark.table(t).count():>9,} rows")

bronze_customers      10,000 rows


bronze_mrr           135,280 rows


bronze_activity      135,280 rows
